In [1]:
import os
import json
import shutil
from dotenv import load_dotenv

from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.document_loaders import TextLoader
from langchain_openai import (
    OpenAIEmbeddings,
    ChatOpenAI,
    OpenAI
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from pathlib import Path
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever
)

from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.retrievers.document_compressors import LLMChainExtractor

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import LocalFileStore, create_kv_docstore

load_dotenv(override=True)



True

In [43]:
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]
EMBEDDING_MODEL_SMALL = "text-embedding-3-small"
CHROMA_PERSIST_DIR = str(PROJECT_ROOT / "data" / "chroma_db")
COLLECTION_NAME = "jd_intelligence_baseline_500d"

In [44]:
db_path = PROJECT_ROOT / "data" / "chroma_db"

if db_path.exists():
    if db_path.is_dir():
        shutil.rmtree(db_path)  
        print("Directory deleted successfully.")
    else:
        os.remove(db_path)     
else:
    print("Path does not exist.")

Directory deleted successfully.


In [47]:
with open(PROJECT_ROOT / "checkpoint.json", 'r', encoding='utf-8') as f:
    checkpoint = json.load(f)

checkpoint

[{'ingested_raw_filenames': ['Accenture_CustomSoftwareEngineer.txt',
   'Careernet_SoftwareEngineer.txt',
   'Data EngineerorETLTester_HCLTech.txt',
   'Digikey_DataEngineer.txt',
   'FirstMeridianBusinessServices_AIDataEngineer.txt',
   'HCLTech_AiEngineer.txt',
   'HimozoTalentConsultingPvtltd_SoftwareEngineer.txt',
   'ImpronicsTechnologies_DataEngineer.txt',
   'Infotrellis_AnalyticsDataBricksEngineer.txt',
   'InfovisionLabs_Data Analyst.txt',
   'InsightGlobalTechnologies_DataAnalyticsEngineer.txt',
   'Kyndryl_DataEngineer.txt',
   'Larsen&Toubro(L&T)_Data Engineer.txt',
   'MystiflyConsulting_DataEngineer.txt',
   'Persistent_JavaDeveloper.txt',
   'RavenrockDevelopments_DataEngineer.txt',
   'Smart Ims_DataEngineer.txt',
   'TataConsultancyServices_DataScientistGenAI.txt',
   'TataConsultancyServices_GoogleDataEngineer.txt ',
   'TechSMCSquaredGCC_DataEngineer.txt',
   'TheBusinessResearchCompany_SeniorDataEngineerDataAnalyticsEngineer.txt',
   'VolvoGroup_ProductQuality&Analy

## **1. Document Loader and Data Normalization**

In [48]:
def load_and_normalize_jd(document_path: Path) -> str:

    loader = TextLoader(document_path)
    documents = loader.load()

    text_data = documents[0].page_content
    # Fixed Syntax
    data = [line for line in text_data.split("\n\n") if line != '\n']

    final_text = []

    for block in data:
        block = block.strip()
        if "\n" in block:
            # Split the sub-lines inside this block
            sub_lines = block.split("\n")
            # Join them back with a single SPACE, filtering out empty entries
            block = ' '.join([sl.strip() for sl in sub_lines if sl.strip()])

        final_text.append(block)

    return ' '.join(final_text)


In [49]:


directory_path = PROJECT_ROOT / "data" / "raw_jds"
normalized_files = []
metadata = {}

for file in os.listdir(directory_path):
    # Track files using the clean string filename ('visa_de.txt')
    if file not in checkpoint[0]["ingested_raw_filenames"]:
        full_file_path = directory_path / file
        
        if full_file_path.is_file():
            normalized_files.append(file)
            print(f"Successfully processed: {file}")

            # Extract, clean, and write the text file
            cleaned_data = load_and_normalize_jd(full_file_path)

            if "Openings:" in cleaned_data:

                metadata["posted_date"] = (
                    cleaned_data.split("Posted:")[1]
                    .split("Openings:")[0]
                    .strip()
                )

                metadata["openings"] = (
                    cleaned_data.split("Openings:")[1]
                    .split("Applicants:")[0]
                    .strip()
                )

            else:

                metadata["posted_date"] = (
                    cleaned_data.split("Posted:")[1]
                    .split("Applicants:")[0]
                    .strip()
                )

                metadata["openings"] = None


            metadata["applicants"] = (
                cleaned_data.split("Applicants:")[1]
                .split("Save")[0]
                .strip()
            )
            
            output_path = PROJECT_ROOT / "data" / "cleaned_jds" / f"{full_file_path.stem}.txt"
            
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(cleaned_data)
            
            # Appending the text filename string so JSON can save it safely
            checkpoint[0]["ingested_raw_filenames"].append(file)

print(f"\nTotal new files processed normalized: {len(normalized_files)}")

with open(PROJECT_ROOT / "checkpoint.json", 'w', encoding='utf-8') as f:
    json.dump(checkpoint, f, indent=4)

print("Checkpoint database updated cleanly.")


Total new files processed normalized: 0
Checkpoint database updated cleanly.


## **2. Text Splitter, Embeddings and Vector Store**

In [50]:
## Text Splitteing or Chunking
def chunk_jd_text(splitter:RecursiveCharacterTextSplitter, source_path:Path) ->  list[Document]:

    cleaned_text = ""

    with open(source_path, 'r', encoding="utf-8") as f:
        cleaned_text = f.read()

    chunks = splitter.create_documents(
        texts=[cleaned_text],
        metadatas=[
            {
                "source" : source_path.stem,
                "Description" : f"{(source_path.stem).split("_")[0]} {(source_path.stem).split("_")[1]} job description"
            }
        ]
    )
    
    return chunks
    
    

In [51]:

cleaned_file_path = Path(PROJECT_ROOT / "data" / "cleaned_jds")

## 1. Text Splitter configuration
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

## 2. Low-dimension Embedding configuration
embeddings_engine = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_SMALL,
    dimensions=500,  # Your target evaluation baseline length
    max_retries=3
)

## 3. Persistent Storage Handler connection
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings_engine,
    persist_directory=CHROMA_PERSIST_DIR
)

files_processed_this_run = 0

all_documents = []

for file in os.listdir(cleaned_file_path):
    if not file.startswith('.') and file.endswith('.txt'):
        
        if file not in checkpoint[0]["normalized_output_filenames"]:
            cleaned_full_file_path = cleaned_file_path / file
            
            # Extract live documents into memory
            chunked_data = chunk_jd_text(splitter=splitter, source_path=cleaned_full_file_path)
            
            if chunked_data:
                # Core fix: add_documents pushes directly to DB and returns a list of chunk IDs
                chunk_ids = vector_store.add_documents(chunked_data)
                all_documents.extend(chunked_data)
                files_processed_this_run += 1

                print(f"==================================================")
                print(f"SUCCESS: {file}")
                print(f" -> Generated Chunks: {len(chunked_data)}")
                print(f" -> Indexed Chunk IDs: {len(chunk_ids)} registered keys")
                print(f" -> Sample ID range: {chunk_ids[0]} ... {chunk_ids[-1]}")
                print(f"==================================================\n")

            # Mark the file as processed in memory state
            checkpoint[0]['normalized_output_filenames'].append(file)

print(f"Processing Batch Complete! {files_processed_this_run} new files indexed safely.")

# 4. Commit tracking records to disk
with open(PROJECT_ROOT / "checkpoint.json", 'w', encoding='utf-8') as f:
    json.dump(checkpoint, f, indent=4)

print("Checkpoint ledger committed safely to checkpoint.json.")

Processing Batch Complete! 0 new files indexed safely.
Checkpoint ledger committed safely to checkpoint.json.


-- not required right now

In [52]:
## Retrival
## 1. Similarity Search
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2
    }
)

query = "Looking for a GCP data engineer Role without Spark or Databricks as a skill"

results_with_scores = vector_store.similarity_search_with_score(query)

print("--- Similarity Search Results with L2 Distance Scores ---")
for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n[Match {i}] Source File: {doc.metadata.get('source')}.txt")
    print(f"    Distance Score: {score:.4f}")  # Smaller = Closer/More accurate match
    print(f"    Content Snippet: {doc.page_content[:150]}...")

--- Similarity Search Results with L2 Distance Scores ---

[Match 0] Source File: TechSMCSquaredGCC_DataEngineer.txt
    Distance Score: 0.6300
    Content Snippet: related field. V. Skill Set Required: Must have: Experience developing data solutions on cloud computing platforms, preferably GCP. 2+ years hands-on ...

[Match 1] Source File: TataConsultancyServices_GoogleDataEngineer.txt
    Distance Score: 0.6355
    Content Snippet: with expertise in BigQuery, NoSQL, ETL tools, and Terraform Design and maintain scalable data pipelines, develop data models, optimize performance, an...

[Match 2] Source File: TataConsultancyServices_GoogleDataEngineer.txt
    Distance Score: 0.6607
    Content Snippet: data needs and deliver solutions that meet their requirements. Desirable Experience Cloud Storage or equivalent cloud platforms Knowledge of BigQuery ...

[Match 3] Source File: Smart Ims_DataEngineer.txt
    Distance Score: 0.6726
    Content Snippet: scalable data pipelines and models i

In [53]:
## Similarity Search with Thershold

threshold = 0.5

retriever_thershould = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": threshold},
)

results = retriever_thershould.invoke(query)

print(f"=== Threshold = {threshold} — {len(results)} document(s) returned ===")
for i, doc in enumerate(results, 1):
    print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")


=== Threshold = 0.5 — 4 document(s) returned ===
  [1] metadata={'applicants': '100+', 'job_title': 'DataEngineer', 'openings': '1', 'description': 'TechSMCSquaredGCC DataEngineer job description', 'source': 'TechSMCSquaredGCC_DataEngineer', 'posted_date': '2 days ago', 'company_name': 'TechSMCSquaredGCC'}: related field. V. Skill Set Required: Must have: Experience developing data solutions on cloud computing platforms, preferably GCP. 2+ years hands-on experience with frameworks such as Python, Java, or Scala. Skills in Python; Google BigQuery, Google Storage, Cloud Composer, Dataflow; Google PubSub; SQL; Ansible; Tableau APIs; Jenkins; and Shell scripting. Nice to have: Experience with Data Warehouse and ETL migrations. Advanced SQL skills and familiarity with various relational databases and
  [2] metadata={'description': 'TataConsultancyServices GoogleDataEngineer job description', 'job_title': 'GoogleDataEngineer', 'applicants': '100+', 'source': 'TataConsultancyServices_GoogleDa

In [54]:
## MMR 

lambda_values = [1.0, 0.7, 0.5 ,0.0]

for lm in lambda_values:
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": lm},
    )
    results = retriever.invoke(query)
    
    topics = [doc.metadata for doc in results]
    print(f"=== lambda_mult={lm} ===")
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] topic={doc.metadata}: {doc.page_content}")
    print()

=== lambda_mult=1.0 ===
  [1] topic={'posted_date': '2 days ago', 'job_title': 'DataEngineer', 'description': 'TechSMCSquaredGCC DataEngineer job description', 'company_name': 'TechSMCSquaredGCC', 'openings': '1', 'source': 'TechSMCSquaredGCC_DataEngineer', 'applicants': '100+'}: related field. V. Skill Set Required: Must have: Experience developing data solutions on cloud computing platforms, preferably GCP. 2+ years hands-on experience with frameworks such as Python, Java, or Scala. Skills in Python; Google BigQuery, Google Storage, Cloud Composer, Dataflow; Google PubSub; SQL; Ansible; Tableau APIs; Jenkins; and Shell scripting. Nice to have: Experience with Data Warehouse and ETL migrations. Advanced SQL skills and familiarity with various relational databases and
  [2] topic={'applicants': '100+', 'posted_date': '3 days ago', 'source': 'TataConsultancyServices_GoogleDataEngineer', 'company_name': 'TataConsultancyServices', 'openings': '1', 'job_title': 'GoogleDataEngineer', 'descr

In [56]:
raw = vector_store.get()
documents = raw.get("documents") or []
metadatas = raw.get("metadatas") or []

if not documents:
    print("Vector store returned zero documents")

if len(metadatas) != len(documents):
    print(
        "Metadata count (%d) does not match document count (%d); "
        "using empty metadata for missing entries",
        len(metadatas),
        len(documents),
    )
    metadatas = metadatas + [{}] * (len(documents) - len(metadatas))

all_documents = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(documents, metadatas)
]
print("Loaded %d document chunk(s) from index", len(all_documents))

bm_retriever = BM25Retriever.from_documents(
    documents=all_documents
)

bm_retriever.k = 3

bm_results = bm_retriever.invoke(
    query
)

bm_results

Loaded %d document chunk(s) from index 197


[Document(metadata={'company_name': 'Infotrellis', 'applicants': 'Less than 10', 'source': 'Infotrellis_AnalyticsDataBricksEngineer', 'posted_date': '3+ weeks ago', 'job_title': 'AnalyticsDataBricksEngineer', 'openings': '1', 'description': 'Infotrellis AnalyticsDataBricksEngineer job description'}, page_content='project deployment and CI/CD. Unity Catalog for data lineage and impact analysis . Spark SQL: CTEs, window functions, MERGE, and SCD patterns . DLT expectations and data quality constraints . Git integration with Databricks Repos for version control . Experience: 3+ years in analytics/data engineering with at least 1 year on Databricks. Delivered 2+ medallion architecture or DLT pipeline projects. Clear communication skills for client interactions and technical documentation. Role: Data Science'),
 Document(metadata={'openings': '1', 'source': 'ImpronicsTechnologies_DataEngineer', 'job_title': 'DataEngineer', 'applicants': '100+', 'posted_date': '3 weeks ago', 'company_name': 

In [57]:
chroma_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

ensemble_retriever = EnsembleRetriever(
    retrievers=[chroma_retriever, bm_retriever],
    weights=[0.8, 0.2]
)

ensemble_results = ensemble_retriever.invoke(
    query
)

for i, doc in enumerate(ensemble_results, 1):
    print(f"  [{i}] {doc.metadata}: {doc.page_content}")

  [1] {'source': 'TechSMCSquaredGCC_DataEngineer', 'applicants': '100+', 'openings': '1', 'description': 'TechSMCSquaredGCC DataEngineer job description', 'company_name': 'TechSMCSquaredGCC', 'job_title': 'DataEngineer', 'posted_date': '2 days ago'}: related field. V. Skill Set Required: Must have: Experience developing data solutions on cloud computing platforms, preferably GCP. 2+ years hands-on experience with frameworks such as Python, Java, or Scala. Skills in Python; Google BigQuery, Google Storage, Cloud Composer, Dataflow; Google PubSub; SQL; Ansible; Tableau APIs; Jenkins; and Shell scripting. Nice to have: Experience with Data Warehouse and ETL migrations. Advanced SQL skills and familiarity with various relational databases and
  [2] {'openings': '1', 'posted_date': '3 days ago', 'company_name': 'TataConsultancyServices', 'description': 'TataConsultancyServices GoogleDataEngineer job description', 'source': 'TataConsultancyServices_GoogleDataEngineer', 'applicants': '100+', 

In [58]:
llm = ChatOpenAI(temperature=0)

compressor = LLMChainExtractor.from_llm(
    llm=llm
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(query)

lambda_values = [0.0]

for lm in lambda_values:
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": lm},
    )
    results = retriever.invoke(query)
    
    topics = [doc.metadata for doc in results]
    print(f"=== lambda_mult={lm} ===")
    for i, doc in enumerate(results, 1):
        print(len(doc.page_content))
        print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")
    print()

print("=" * 50)

print(len(compressed_docs[0].page_content))
print(compressed_docs)

=== lambda_mult=0.0 ===
497
  [1] metadata={'description': 'TechSMCSquaredGCC DataEngineer job description', 'source': 'TechSMCSquaredGCC_DataEngineer', 'job_title': 'DataEngineer', 'company_name': 'TechSMCSquaredGCC', 'openings': '1', 'applicants': '100+', 'posted_date': '2 days ago'}: related field. V. Skill Set Required: Must have: Experience developing data solutions on cloud computing platforms, preferably GCP. 2+ years hands-on experience with frameworks such as Python, Java, or Scala. Skills in Python; Google BigQuery, Google Storage, Cloud Composer, Dataflow; Google PubSub; SQL; Ansible; Tableau APIs; Jenkins; and Shell scripting. Nice to have: Experience with Data Warehouse and ETL migrations. Advanced SQL skills and familiarity with various relational databases and
486
  [2] metadata={'source': 'Larsen&Toubro(L&T)_Data Engineer', 'description': 'Larsen&Toubro(L&T) Data Engineer job description', 'company_name': 'Larsen&Toubro(L&T)', 'openings': '5', 'job_title': 'Data Enginee

### **Parent Document Retriever**

In [60]:
## For child Chunks
vector_store_parent_child = Chroma(
    collection_name="split_parent",
    embedding_function=embeddings_engine,
    persist_directory=CHROMA_PERSIST_DIR
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 70
)

## For parent chunks
fs = LocalFileStore(root_path=Path(PROJECT_ROOT / "data" / "doc_store"))

persistent_docstore = create_kv_docstore(fs)

parent_retirever = ParentDocumentRetriever(
    vectorstore=vector_store_parent_child,
    docstore=persistent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)



cleaned_data = []

for file in os.listdir(cleaned_file_path):

    if os.path.exists(Path(PROJECT_ROOT / "data" / "cleaned_jds" / file)):
       
        with open(Path(PROJECT_ROOT / "data" / "cleaned_jds" / file), 'r', encoding='utf-8') as f:
            cleaned_data.extend([Document(
                page_content=f.read(),
                metadata={
                        "source" : Path(file).stem,
                        "Description" : Path(file).stem.replace("_", " ") + " job description"
                }
            )]
            )

    
parent_results = parent_retirever.add_documents(cleaned_data)

parent_keys = list(persistent_docstore.yield_keys())
total_parents = len(parent_keys)

print(f"Total Parent Chunks generated and stored: {total_parents}")

print()

# Query ChromaDB for the total number of items in this specific collection
total_children = vector_store_parent_child._collection.count()

print(f"Total Child Chunks generated and indexed: {total_children}")


Total Parent Chunks generated and stored: 144

Total Child Chunks generated and indexed: 1072


In [61]:

parent_retriever = vector_store_parent_child.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.3},
)


parent_retrieve_results = parent_retriever.invoke(query)

print(parent_retrieve_results)

[Document(metadata={'Description': 'Smart Ims DataEngineer job description', 'doc_id': '86443276-d97e-486d-96b8-237d1eac6199', 'source': 'Smart Ims_DataEngineer'}, page_content='pipelines and models in BigQuery; ensure data quality and automate ETL processes using GCP services; troubleshoot and collaborate with cross-functional teams Job match score Early Applicant Keyskills Location Work Experience Job description Job description Role: Data Engineer Experience: 3-7 Years'), Document(metadata={'Description': 'TechSMCSquaredGCC DataEngineer job description', 'doc_id': '09c31855-9574-47c3-b61c-2560d161c33a', 'source': 'TechSMCSquaredGCC_DataEngineer'}, page_content='V. Skill Set Required: Must have: Experience developing data solutions on cloud computing platforms, preferably GCP. 2+ years hands-on experience with frameworks such as Python, Java, or Scala. Skills in Python; Google BigQuery, Google Storage, Cloud Composer, Dataflow; Google PubSub; SQL; Ansible;'), Document(metadata={'doc_

In [62]:
topics = [doc.metadata for doc in parent_retrieve_results]
print(f"=== lambda_mult={0.3} ===")
for i, doc in enumerate(parent_retrieve_results, 1):
    print(len(doc.page_content))
    print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")

=== lambda_mult=0.3 ===
298
  [1] metadata={'Description': 'Smart Ims DataEngineer job description', 'doc_id': '86443276-d97e-486d-96b8-237d1eac6199', 'source': 'Smart Ims_DataEngineer'}: pipelines and models in BigQuery; ensure data quality and automate ETL processes using GCP services; troubleshoot and collaborate with cross-functional teams Job match score Early Applicant Keyskills Location Work Experience Job description Job description Role: Data Engineer Experience: 3-7 Years
299
  [2] metadata={'Description': 'TechSMCSquaredGCC DataEngineer job description', 'doc_id': '09c31855-9574-47c3-b61c-2560d161c33a', 'source': 'TechSMCSquaredGCC_DataEngineer'}: V. Skill Set Required: Must have: Experience developing data solutions on cloud computing platforms, preferably GCP. 2+ years hands-on experience with frameworks such as Python, Java, or Scala. Skills in Python; Google BigQuery, Google Storage, Cloud Composer, Dataflow; Google PubSub; SQL; Ansible;
293
  [3] metadata={'doc_id': '80

In [64]:
for i, doc in enumerate(parent_retirever.invoke(query), 1):
    print(len(doc.page_content))
    print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")

In [65]:
# Assuming 'db' is your LangChain Chroma instance
results = vector_store.get(include=['embeddings', 'documents', 'metadatas'])

# Access the data
vectors = results['embeddings']
documents = results['documents']
ids = results['ids']


### **Rag Fusion**

In [2]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=500)
db = Chroma(
    collection_name="jd_intelligence_baseline_500d",
    embedding_function=embeddings_model,
    persist_directory="/Users/lokeshkv/data-engineering/projects/jd-intelligence-system/data/chroma_db"
)

base_retriever = db.as_retriever(
    search_kwargs={
        'k': 3  
    }
)

/var/folders/gp/0r9x4vj97xn9cbjqk5n01ykc0000gn/T/ipykernel_25038/835427274.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


In [3]:
def generate_sub_queries(original_query: str, llm: ChatOpenAI) -> list[str]:
    prompt_template = ChatPromptTemplate.from_messages(
        [
            (
                "system", 
                "You are an AI assistant that generates multiple search query variations based on a single input query. "
                "Generate exactly 3 distinct variations, each focusing on different keywords or angles of the query. "
                "Output each query on a new line. Do not number them.",
            ),
            (
                "human",
                "{query}"
            )
        ]
    )

    formatted_messages = prompt_template.format_messages(query=original_query)

    response = llm.invoke(formatted_messages)

    queries = [line.strip() for line in response.content.strip().split("\n") if line.strip()]

    if original_query not in queries:
        queries.append(f"- {original_query}")
    
    return queries

In [5]:
llm = ChatOpenAI(temperature=0)
query = "Looking for a GCP data engineer Role without Spark or Databricks as a skill"

queries =generate_sub_queries(
    original_query=query,
    llm=llm
)


In [7]:
import hashlib

rrf_scores = {}
doc_mapping = {}

CONSTANT_K = 60
TOP_K_PER_QUERY = 5

for q in queries:

    docs = base_retriever.invoke(q)

    print(f"\nQuery: {q}")

    for rank, doc in enumerate(docs[:TOP_K_PER_QUERY], start=1):

        # Stable document identifier
        doc_id = hashlib.md5(
            doc.page_content.encode("utf-8")
        ).hexdigest()

        # Store document once
        if doc_id not in doc_mapping:
            doc_mapping[doc_id] = doc

        # RRF score
        score = 1.0 / (CONSTANT_K + rank)

        rrf_scores[doc_id] = (
            rrf_scores.get(doc_id, 0) + score
        )

        print(
            f"Rank={rank} | "
            f"RRF Score={score:.6f} | "
            f"Company={doc.metadata.get('company_name')}"
        )

# Sort by final RRF score
sorted_doc_ids = sorted(
    rrf_scores.items(),
    key=lambda item: item[1],
    reverse=True
)

# Final reranked documents
reranked_docs = [
    doc_mapping[doc_id]
    for doc_id, _ in sorted_doc_ids
]

# Display ranking
print("\n===== FINAL RRF RANKING =====")

for i, (doc_id, score) in enumerate(sorted_doc_ids, start=1):

    doc = doc_mapping[doc_id]

    print(
        f"{i}. "
        f"{doc.metadata.get('company_name')} | "
        f"{doc.metadata.get('job_title')} | "
        f"RRF Score={score:.6f}"
    )

seen_jobs = set()
final_docs = []

for doc in reranked_docs:

    job_key = (
        doc.metadata.get("company_name"),
        doc.metadata.get("job_title")
    )

    if job_key in seen_jobs:
        continue

    seen_jobs.add(job_key)
    final_docs.append(doc)

reranked_docs = final_docs


Query: - GCP data engineer job without Spark or Databricks experience
Rank=1 | RRF Score=0.016393 | Company=TataConsultancyServices
Rank=2 | RRF Score=0.016129 | Company=TechSMCSquaredGCC
Rank=3 | RRF Score=0.015873 | Company=Smart Ims

Query: - Data engineering role on GCP excluding Spark and Databricks requirement
Rank=1 | RRF Score=0.016393 | Company=TataConsultancyServices
Rank=2 | RRF Score=0.016129 | Company=TechSMCSquaredGCC
Rank=3 | RRF Score=0.015873 | Company=TataConsultancyServices

Query: - GCP data engineer position seeking candidates without Spark or Databricks proficiency
Rank=1 | RRF Score=0.016393 | Company=Larsen&Toubro(L&T)
Rank=2 | RRF Score=0.016129 | Company=TechSMCSquaredGCC
Rank=3 | RRF Score=0.015873 | Company=TataConsultancyServices

Query: - Looking for a GCP data engineer Role without Spark or Databricks as a skill
Rank=1 | RRF Score=0.016393 | Company=TataConsultancyServices
Rank=2 | RRF Score=0.016129 | Company=TechSMCSquaredGCC
Rank=3 | RRF Score=0.01587

In [8]:
reranked_docs

[Document(metadata={'applicants': '100+', 'description': 'TechSMCSquaredGCC DataEngineer job description', 'company_name': 'TechSMCSquaredGCC', 'job_title': 'DataEngineer', 'openings': '1', 'source': 'TechSMCSquaredGCC_DataEngineer', 'posted_date': '2 days ago'}, page_content='related field. V. Skill Set Required: Must have: Experience developing data solutions on cloud computing platforms, preferably GCP. 2+ years hands-on experience with frameworks such as Python, Java, or Scala. Skills in Python; Google BigQuery, Google Storage, Cloud Composer, Dataflow; Google PubSub; SQL; Ansible; Tableau APIs; Jenkins; and Shell scripting. Nice to have: Experience with Data Warehouse and ETL migrations. Advanced SQL skills and familiarity with various relational databases and'),
 Document(metadata={'posted_date': '3 days ago', 'company_name': 'TataConsultancyServices', 'description': 'TataConsultancyServices GoogleDataEngineer job description', 'source': 'TataConsultancyServices_GoogleDataEnginee